## <center><b> Pre-Processing MST data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pyvital
from dotenv import load_dotenv
import os
from tqdm import tqdm
import statsmodels.api as sm

In [2]:
# Setup
load_dotenv('vars.env')
DATA_PATH = os.getenv("DATA_PATH")
SAMPLE_LENGTH = int(os.getenv("SAMPLE_LENGTH"))
SAMPLING_RATE = float(os.getenv("SAMPLING_RATE"))

PLOT_KOPPELING = os.getenv("TOON_PLOT_KOPPELING") == "True"
EIND_PLOT = os.getenv("TOON_EIND_PLOT") == "True"

FILTER_RUIS = os.getenv("FILTER_RUIS") == "True"

In [3]:
# Paden van de patiëntenbestanden verzamelen
patienten = []
for folder in sorted(os.listdir(DATA_PATH)):
    if folder.endswith('_vital'):
        patient_id = folder.replace('_vital', '')
        vital_pad  = os.path.join(DATA_PATH, folder, 'vital.csv')
        hs_pad     = os.path.join(DATA_PATH, patient_id + '_hs', 'hemosphere.csv')

        if os.path.exists(vital_pad) and os.path.exists(hs_pad):
            patienten.append((vital_pad, hs_pad))

print(f"Gevonden patiënten: {len(patienten)}")
for v, h in patienten:
    print(f"Vital: {v}, Hemosphere: {h}")

Gevonden patiënten: 2
Vital: /Users/ruben/Documents/Universiteit/Studiejaar 3/Module 12 : TGO/Data/Testdata_pt_5_6_MST kopie/Test_data/Data/MARTINI_110006_vital/vital.csv, Hemosphere: /Users/ruben/Documents/Universiteit/Studiejaar 3/Module 12 : TGO/Data/Testdata_pt_5_6_MST kopie/Test_data/Data/MARTINI_110006_hs/hemosphere.csv
Vital: /Users/ruben/Documents/Universiteit/Studiejaar 3/Module 12 : TGO/Data/Testdata_pt_5_6_MST kopie/Test_data/Data/MARTINI_110007_vital/vital.csv, Hemosphere: /Users/ruben/Documents/Universiteit/Studiejaar 3/Module 12 : TGO/Data/Testdata_pt_5_6_MST kopie/Test_data/Data/MARTINI_110007_hs/hemosphere.csv


In [ ]:
# Data inladen
def laad_vital(pad):
    df_v = pd.read_csv(pad)
    df_v['Time']    = pd.to_timedelta(df_v['Time'])
    df_v['minutes'] = (df_v['Time'] - df_v['Time'].min()).dt.total_seconds() / 60
    return df_v

def laad_hemosphere(pad):
    df_h = pd.read_csv(pad, sep=';')
    df_h = df_h.iloc[1:].reset_index(drop=True)
    df_h['Tijd'] = pd.to_timedelta(df_h['Tijd'].str.strip())
    df_h['SV'] = pd.to_numeric(df_h['SV'].str.strip().replace(r'\\n', np.nan, regex=True), errors='coerce') #verwijderd de \n tekens in de SV kolom.
    df_h = df_h.dropna(subset=['SV']).reset_index(drop=True)
    return df_h

# ABP resampling
def resample_abp(df):
    ABP = df[['minutes', 'Intellivue/ABP']].dropna().reset_index(drop=True)
    ABP.columns = ['minutes', 'ABP']
    time_diff = ABP['minutes'].diff().dropna() * 60
    sample_freq = 1 / time_diff.mean()
    resample_ABP = np.array(pyvital.resample_hz(ABP['ABP'], sample_freq, 100), dtype=np.float32)
    t_start = ABP['minutes'].min()
    return resample_ABP, t_start

# Koppeling van ABP en SV
def koppel_abp_sv(df, df_h, resample_ABP, toon_plot=PLOT_KOPPELING):
    time_vital = df['Time'].dt.total_seconds().values
    time_hs    = df_h['Tijd'].dt.total_seconds().values
    sv_values  = df_h['SV'].values
    t_abp_klok_start = time_vital[0]
    t_abp_klok_end   = time_vital[-1]

    samples_abp, samples_sv, sv_times = [], [], []

    for i in range(len(time_hs)):
        sv_time = time_hs[i]

        if sv_time < t_abp_klok_start or sv_time > t_abp_klok_end:
            continue

        sv_idx    = round((sv_time - t_abp_klok_start) * 100)
        start_idx = sv_idx - SAMPLE_LENGTH

        if start_idx < 0 or sv_idx > len(resample_ABP):
            continue

        segment = np.array(resample_ABP[start_idx:sv_idx])
        if len(segment) != SAMPLE_LENGTH:
            continue

        samples_abp.append(segment)
        samples_sv.append(sv_values[i])
        sv_times.append(sv_time)

    samples_abp = np.array(samples_abp, dtype=np.float32)
    samples_sv  = np.array(samples_sv)
    sv_times    = np.array(sv_times)
    g_indices   = ((sv_times - t_abp_klok_start) / 20).astype(int)

    if toon_plot:
        fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

        for i in range(len(samples_abp)):
            t_seg = np.linspace(sv_times[i] - 20, sv_times[i], SAMPLE_LENGTH) / 3600
            axes[0].plot(t_seg, samples_abp[i], color="#85B7EB", linewidth=0.5)
            axes[0].axvline(sv_times[i] / 3600, color="#E24B4A", linewidth=0.5, alpha=0.3)

        axes[0].set_title("ABP segmenten met SV tijdstippen (rood)", fontsize=11, fontweight="bold", loc="left")
        axes[0].set_ylabel("ABP (mmHg)")
        axes[0].grid(True, alpha=0.2)

        axes[1].plot(sv_times / 3600, samples_sv, color="#378ADD", linewidth=1.0, marker="o", markersize=3)
        axes[1].set_title("Slagvolume (SV)", fontsize=11, fontweight="bold", loc="left")
        axes[1].set_ylabel("SV (mL)")
        axes[1].set_xlabel("Kloktijd (uur)")
        axes[1].grid(True, alpha=0.2)

        plt.tight_layout()
        plt.show()

    return samples_abp, samples_sv, sv_times, g_indices

# Lowess smoothing toepassen op SV waarden
def lowess_smoothing(samples_sv, indices_sv, sampling_rate=1 / 100): #Filter vanuit, van Mierlo
    """
    Performs LOWESS smoothing using the statsmodels.nonparametric.smoothers_lowess.lowess package. If SV values were
    not recorded for > 200s within a single record, the LOWESS algorithm is applied to the slices before and after the
    measurement gap separately.
    :param samples_sv:np.ndarray           SVs at the end of the corresponding ABP segment with shape (nr_of_samples, 1)
    :param indices_sv:np.ndarray           time indices of the SVs with shape (nr_of_samples, 1)
    :param sampling_rate:float             sampling rate of the data in seconds
    :return:
            samples_sv_smoothed:np.ndarray SVs smoothed using LOWESS with shape (nr_of_samples, 1)
    """
    diffs = np.diff(indices_sv.reshape(-1))
    slicing_locs = (np.where(diffs > 200 / sampling_rate)[0] + 1).tolist()  # +1 since we want the slice loc
    slicing_locs = [0] + slicing_locs + [len(indices_sv) + 1]  # add start and end of list
    samples_sv_smoothed = np.array([])
    for slice_index in range(1, len(slicing_locs)):  # loop over the slices
        i_slice = indices_sv[slicing_locs[slice_index - 1]:slicing_locs[slice_index]]
        s_slice = samples_sv[slicing_locs[slice_index - 1]:slicing_locs[slice_index]]
        s_slice_smoothed = sm.nonparametric.lowess(exog=i_slice.reshape([len(i_slice)]),
                                                   endog=s_slice.reshape([len(s_slice)]),
                                                   frac=0.03, return_sorted=False)
        samples_sv_smoothed = np.append(samples_sv_smoothed, s_slice_smoothed)
    return samples_sv_smoothed.reshape(len(samples_sv_smoothed), 1)

def lowess_sv(samples_sv, g_indices): #Toepassing filter van Mierlo
    indices_sv = (g_indices * 2000).reshape(-1, 1)
    return lowess_smoothing(
        samples_sv=samples_sv.reshape(-1, 1),
        indices_sv=indices_sv,
        sampling_rate=SAMPLING_RATE
    )

# Ruis filtering
def filter_fysiologisch(segments, sv_smoothed, indices_to_be_removed): #Fysiologische onwaarheden, zoals ABP < 25 of > 250 mmHg, of SV < 20 of > 200 mL. 
    mask = (
        (segments.min(axis=1) < 25) | (segments.max(axis=1) > 250) |
        (sv_smoothed.flatten() < 20) | (sv_smoothed.flatten() > 200)
    )
    indices_to_be_removed.update(np.where(mask)[0])

# Extra ruis filtering
def detect_unrealistic_segment(segment, max_step=30, max_noise=5): #Geeft True of False terug, nodig voor  def filter_ruis
    diff = np.abs(np.diff(segment))
    return diff.max() > max_step or diff.std() > max_noise

def filter_ruis(segments, indices_to_be_removed, actief=FILTER_RUIS):
    if not actief:
        return
    mask = np.array([detect_unrealistic_segment(seg) for seg in segments])
    indices_to_be_removed.update(np.where(mask)[0])
    
# Segmenten verwijderen
def verwijder_segmenten(segments, sv, g_indices, sv_times, indices_to_be_removed, patient_naam="", toon_plot=EIND_PLOT):
    mask = np.ones(len(segments), dtype=bool)
    mask[list(indices_to_be_removed)] = False

    if toon_plot:
        fig, axes = plt.subplots(2, 1, figsize=(16, 8))

        # Voor filtering — segmenten aaneengesloten, rood = verwijderd
        for i in range(len(segments)):
            x = np.arange(i * SAMPLE_LENGTH, (i + 1) * SAMPLE_LENGTH) / 100 / 60  # minuten
            kleur = "#E24B4A" if not mask[i] else "#85B7EB"
            axes[0].plot(x, segments[i], color=kleur, linewidth=0.4, alpha=0.8)

        axes[0].plot([], [], color="#85B7EB", label="Goed segment")
        axes[0].plot([], [], color="#E24B4A", label="Verwijderd segment")
        axes[0].set_title(f"{patient_naam} — Voor filtering", fontsize=11, fontweight="bold", loc="left")
        axes[0].set_ylabel("ABP (mmHg)")
        axes[0].set_xlabel("Tijd (min)")
        axes[0].legend(fontsize=9)
        axes[0].grid(True, alpha=0.2)

        # Na filtering — alleen goede segmenten aaneengesloten
        goede_segmenten = segments[mask]
        for i in range(len(goede_segmenten)):
            x = np.arange(i * SAMPLE_LENGTH, (i + 1) * SAMPLE_LENGTH) / 100 / 60
            axes[1].plot(x, goede_segmenten[i], color="#85B7EB", linewidth=0.4)

        axes[1].set_title(f"{patient_naam} — Na filtering", fontsize=11, fontweight="bold", loc="left")
        axes[1].set_ylabel("ABP (mmHg)")
        axes[1].set_xlabel("Tijd (min)")
        axes[1].grid(True, alpha=0.2)

        plt.tight_layout()
        plt.show()

    return segments[mask], sv[mask], g_indices[mask]

#Speelt alle defenities af
def verwerk_patient(vital_pad, hs_pad):
    patient_naam = os.path.basename(os.path.dirname(vital_pad))
    print(f'Verwerken: {vital_pad}')

    df_v   = laad_vital(vital_pad)
    df_h = laad_hemosphere(hs_pad)

    resample_ABP, t_start = resample_abp(df_v)

    segments, sv, sv_times, g_indices = koppel_abp_sv(df_v, df_h, resample_ABP)
    print(f'Gekoppeld: {len(segments)}')

    sv_smoothed = lowess_sv(sv, g_indices)

    indices_to_be_removed = set()

    stappen = [
        ('Fysiologisch',   lambda: filter_fysiologisch(segments, sv_smoothed, indices_to_be_removed)),
        ('Ruis',           lambda: filter_ruis(segments, indices_to_be_removed)),
        #('Hartslag',       lambda: filter_hartslag(segments, indices_to_be_removed)),
        #('Pulse pressure', lambda: filter_pulse_pressure(segments, indices_to_be_removed)),
    ]

    with tqdm(total=len(stappen), desc='  Filtering') as pbar: #pbar en vooruitgangs weergave.
        for naam, stap in stappen:
            voor = len(indices_to_be_removed)
            stap()
            na = len(indices_to_be_removed)
            pbar.set_postfix({naam: f'+{na - voor}'})
            pbar.update(1)

    filtered_segments, filtered_sv, filtered_indices = verwijder_segmenten(segments, sv_smoothed.flatten(), sv_times, g_indices, indices_to_be_removed, patient_naam=patient_naam)

    print(f'  Na filtering:       {len(filtered_segments)}')
    print(f'  Verwijderd totaal:  {len(indices_to_be_removed)}')

    return filtered_segments, filtered_sv, filtered_indices

In [8]:
%matplotlib qt
alle_segmenten = []
alle_sv        = []
alle_indices   = []

with tqdm(total=len(patienten), desc='Patiënten verwerken') as pbar:
    for vital_pad, hs_pad in patienten:
        seg, sv, idx = verwerk_patient(vital_pad, hs_pad)
        alle_segmenten.append(seg)
        alle_sv.append(sv)
        alle_indices.append(idx)
        pbar.update(1)

alle_segmenten = np.concatenate(alle_segmenten, axis=0)
alle_sv        = np.concatenate(alle_sv, axis=0)
alle_indices   = np.concatenate(alle_indices, axis=0)

print(f'\nTotaal segmenten: {len(alle_segmenten)}')
print(f'Totaal SV:        {len(alle_sv)}')

Patiënten verwerken:   0%|          | 0/2 [00:00<?, ?it/s]

Verwerken: /Users/ruben/Documents/Universiteit/Studiejaar 3/Module 12 : TGO/Data/Testdata_pt_5_6_MST kopie/Test_data/Data/MARTINI_110006_vital/vital.csv
Gekoppeld: 1693


Patiënten verwerken:  50%|█████     | 1/2 [01:32<01:32, 92.31s/it]

  Na filtering:       1644
  Verwijderd totaal:  49
Verwerken: /Users/ruben/Documents/Universiteit/Studiejaar 3/Module 12 : TGO/Data/Testdata_pt_5_6_MST kopie/Test_data/Data/MARTINI_110007_vital/vital.csv
Gekoppeld: 199


Patiënten verwerken: 100%|██████████| 2/2 [01:35<00:00, 47.68s/it]

  Na filtering:       194
  Verwijderd totaal:  5

Totaal segmenten: 1838
Totaal SV:        1838
